In [ ]:
!pip install -q segmentation-models-pytorch albumentations torchmetrics pytorch-lightning clearml python-dotenv gdown

In [ ]:
import os
import math
import shutil
import torch
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
import segmentation_models_pytorch as smp
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, TQDMProgressBar
from torch.utils.data import DataLoader, Dataset
from torchmetrics import JaccardIndex
from albumentations.pytorch import ToTensorV2
from clearml import Task
import cv2
import numpy as np
from google.colab import files
from dotenv import load_dotenv
import gdown
import zipfile
from concurrent.futures import ProcessPoolExecutor

In [ ]:
from google.colab import userdata, drive

os.environ['CLEARML_API_ACCESS_KEY'] = userdata.get('CLEARML_API_ACCESS_KEY')
os.environ['CLEARML_API_SECRET_KEY'] = userdata.get('CLEARML_API_SECRET_KEY')

In [ ]:
drive.mount('/content/drive')


source_path = '/content/drive/MyDrive/'

tasks = [
    {
        "files": ["leftImg8bit_trainvaltest.zip", "gtFine_trainvaltest.zip"],
        "dest": "/content/data/cityscapes/"
    }
]

for task in tasks:
    if not os.path.exists(task["dest"]):
        os.makedirs(task["dest"])

    for zip_file in task["files"]:
        full_path = os.path.join(source_path, zip_file)

        if os.path.exists(full_path):
            print(f"Unpacking {zip_file} in {task['dest']}...")
            !unzip -q -o "{full_path}" -d "{task['dest']}"
        else:
            print(f"Error, file not found: {full_path}")

In [ ]:
CONFIG = {
    "project_name": "Segmentation_Urban_Scene_CourseWork",
    "task_name": "SegFormer_B2_Cityscapes_E3_FrequencyStyleAug",
    "data_dir": "./data/cityscapes",
    "encoder": "mit_b2",
    "encoder_weights": "imagenet",
    "classes": 19,
    "batch_size": 8,
    "lr": 1e-4,
    "momentum": 0.9,
    "weight_decay": 0.01,
    "epochs": 50,
    "image_size": (512, 1024),
    "frequency_aug_p": 0.35,
    "frequency_beta_range": (0.01, 0.05),
    "frequency_blend_range": (0.3, 0.7),
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# Set precision for Tensor Cores
torch.set_float32_matmul_precision('medium')

# Set allocator to avoid fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

task = Task.init(
    project_name=CONFIG["project_name"],
    task_name=CONFIG["task_name"],
    output_uri=False
)
task.connect(CONFIG)

In [ ]:
def fourier_low_freq_mix(content_image, style_image, beta=0.03, blend=0.5):
    """FDA-inspired source-only style mix: keep content structure, borrow low-frequency style."""
    if style_image.shape[:2] != content_image.shape[:2]:
        style_image = cv2.resize(style_image, (content_image.shape[1], content_image.shape[0]), interpolation=cv2.INTER_LINEAR)

    content = content_image.astype(np.float32).transpose(2, 0, 1)
    style = style_image.astype(np.float32).transpose(2, 0, 1)

    fft_content = np.fft.fft2(content, axes=(-2, -1))
    fft_style = np.fft.fft2(style, axes=(-2, -1))

    amp_content = np.abs(fft_content)
    phase_content = np.angle(fft_content)
    amp_style = np.abs(fft_style)

    amp_content = np.fft.fftshift(amp_content, axes=(-2, -1))
    amp_style = np.fft.fftshift(amp_style, axes=(-2, -1))

    h, w = amp_content.shape[-2:]
    band = max(1, int(np.floor(min(h, w) * beta)))
    c_h, c_w = h // 2, w // 2
    h1, h2 = c_h - band, c_h + band + 1
    w1, w2 = c_w - band, c_w + band + 1

    amp_content[:, h1:h2, w1:w2] = (
        (1.0 - blend) * amp_content[:, h1:h2, w1:w2]
        + blend * amp_style[:, h1:h2, w1:w2]
    )

    amp_content = np.fft.ifftshift(amp_content, axes=(-2, -1))
    mixed = np.fft.ifft2(amp_content * np.exp(1j * phase_content), axes=(-2, -1)).real
    mixed = np.clip(mixed.transpose(1, 2, 0), 0, 255).astype(np.uint8)
    return mixed


def apply_source_only_frequency_aug(content_image, style_image, beta_range=(0.01, 0.05), blend_range=(0.3, 0.7)):
    beta = float(np.random.uniform(*beta_range))
    blend = float(np.random.uniform(*blend_range))
    return fourier_low_freq_mix(content_image, style_image, beta=beta, blend=blend)


class CityscapesDataset(Dataset):
    def __init__(self, root_dir, split='train', augmentation=None, preprocessing=None, frequency_aug_p=0.0, frequency_beta_range=(0.01, 0.05), frequency_blend_range=(0.3, 0.7)):
        self.root_dir = root_dir
        self.images_dir = os.path.join(root_dir, 'leftImg8bit', split)
        self.masks_dir = os.path.join(root_dir, 'gtFine', split)
        self.augmentation = augmentation
        self.preprocessing = preprocessing
        self.frequency_aug_p = frequency_aug_p if split == "train" else 0.0
        self.frequency_beta_range = frequency_beta_range
        self.frequency_blend_range = frequency_blend_range
        self.ids = []

        for city in os.listdir(self.images_dir):
            img_dir = os.path.join(self.images_dir, city)
            mask_dir = os.path.join(self.masks_dir, city)
            for file_name in os.listdir(img_dir):
                if file_name.endswith('_leftImg8bit.png'):
                   self.ids.append({
                       'image': os.path.join(img_dir, file_name),
                       'mask': os.path.join(mask_dir, file_name.replace('_leftImg8bit.png', '_gtFine_labelIds.png'))
                   })

        self.mapping = {
            7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
            23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18
        }

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        sample = self.ids[i]
        image = cv2.imread(sample['image'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.frequency_aug_p > 0 and len(self.ids) > 1 and np.random.rand() < self.frequency_aug_p:
            style_idx = np.random.randint(0, len(self.ids) - 1)
            if style_idx >= i:
                style_idx += 1
            style_image = cv2.imread(self.ids[style_idx]['image'])
            style_image = cv2.cvtColor(style_image, cv2.COLOR_BGR2RGB)
            image = apply_source_only_frequency_aug(
                image,
                style_image,
                beta_range=self.frequency_beta_range,
                blend_range=self.frequency_blend_range,
            )

        mask = cv2.imread(sample['mask'], 0)

        mask_mapped = np.ones_like(mask) * 255
        for k, v in self.mapping.items():
            mask_mapped[mask == k] = v
        mask = mask_mapped

        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        if self.preprocessing:
            sample = self.preprocessing(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        return image, mask.long()

In [ ]:
def _to_float32(image, **kwargs):
    return image.astype(np.float32)


class RoadDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size, image_size, encoder, encoder_weights, frequency_aug_p=0.0, frequency_beta_range=(0.01, 0.05), frequency_blend_range=(0.3, 0.7)):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.image_size = image_size
        self.frequency_aug_p = frequency_aug_p
        self.frequency_beta_range = frequency_beta_range
        self.frequency_blend_range = frequency_blend_range
        self.preprocessing_fn = smp.encoders.get_preprocessing_fn(encoder, encoder_weights)

    def setup(self, stage=None):
        self.train_transform = A.Compose([
            A.Resize(height=self.image_size[0], width=self.image_size[1]),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=10, p=0.5),
            A.OneOf([
                A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
                A.RandomGamma(gamma_limit=(70, 120), p=1.0),
                A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=1.0),
                A.RandomShadow(p=1.0),
            ], p=0.75),
            A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.08, p=0.4),
            A.OneOf([
                A.RandomFog(p=1.0),
                A.RandomRain(blur_value=3, brightness_coefficient=0.9, p=1.0),
                A.RandomSnow(p=1.0),
            ], p=0.35),
            A.OneOf([
                A.GaussNoise(p=1.0),
                A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                A.MotionBlur(blur_limit=5, p=1.0),
            ], p=0.25),
        ])

        self.valid_transform = A.Compose([
            A.Resize(height=self.image_size[0], width=self.image_size[1]),
        ])

        self.preprocessing = A.Compose([
            A.Lambda(image=self.preprocessing_fn),
            A.Lambda(image=_to_float32),
            A.pytorch.ToTensorV2()
        ])

        self.train_ds = CityscapesDataset(self.data_dir, split='train', augmentation=self.train_transform, preprocessing=self.preprocessing, frequency_aug_p=self.frequency_aug_p, frequency_beta_range=self.frequency_beta_range, frequency_blend_range=self.frequency_blend_range)
        self.val_ds = CityscapesDataset(self.data_dir, split='val', augmentation=self.valid_transform, preprocessing=self.preprocessing)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=2, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
from torch.optim.lr_scheduler import SequentialLR, LinearLR, PolynomialLR

class ClearMLMetricsCallback(Callback):
    def __init__(self, task):
        super().__init__()
        self._logger = task.get_logger()

    def _log_metrics(self, trainer, stage):
        for name, value in trainer.callback_metrics.items():
            if isinstance(value, torch.Tensor):
                value = value.detach().cpu().item()
            if isinstance(value, (int, float)) and math.isfinite(value):
                self._logger.report_scalar(stage, name, value, iteration=trainer.current_epoch)

    def on_train_epoch_end(self, trainer, pl_module):
        self._log_metrics(trainer, "train")

    def on_validation_epoch_end(self, trainer, pl_module):
        self._log_metrics(trainer, "val")


class RoadModel(pl.LightningModule):
    def __init__(self, encoder, encoder_weights, in_channels, out_classes, lr=1e-4, weight_decay=0.01, epochs=50):
        super().__init__()
        self.save_hyperparameters()

        self.model = smp.Segformer(
            encoder_name=encoder,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=out_classes,
        )

        self.loss_fn = torch.nn.CrossEntropyLoss(ignore_index=255)

        self.train_miou = JaccardIndex(task="multiclass", num_classes=out_classes, ignore_index=255)
        self.val_miou = JaccardIndex(task="multiclass", num_classes=out_classes, ignore_index=255)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)

        loss = self.loss_fn(logits, masks)

        preds = torch.argmax(logits, dim=1)
        self.train_miou(preds, masks)

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log("train_miou", self.train_miou, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss = self.loss_fn(logits, masks)

        preds = torch.argmax(logits, dim=1)
        self.val_miou(preds, masks)

        self.log("val_loss", loss, on_epoch=True, prog_bar=True, logger=True)
        self.log("val_miou", self.val_miou, on_epoch=True, prog_bar=True, logger=True)

        if batch_idx == 0:
            experiment = getattr(self.logger, "experiment", None)
            if experiment is not None and hasattr(experiment, "report_image"):
                experiment.report_image(
                    "Validation Prediction",
                    "Image_0",
                    iteration=self.current_epoch,
                    image=preds[0].float().cpu().numpy() / 19.0 * 255
                )

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
            betas=(0.9, 0.999),
            eps=1e-8
        )

        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = 1500

        decay_steps = total_steps - warmup_steps
        if decay_steps < 0:
            decay_steps = 1

        scheduler1 = LinearLR(
            optimizer,
            start_factor=0.01,
            end_factor=1.0,
            total_iters=warmup_steps
        )

        scheduler2 = PolynomialLR(
            optimizer,
            total_iters=decay_steps,
            power=0.9
        )

        scheduler = SequentialLR(
            optimizer,
            schedulers=[scheduler1, scheduler2],
            milestones=[warmup_steps]
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1
            }
        }

In [ ]:
dm = RoadDataModule(
    CONFIG["data_dir"],
    CONFIG["batch_size"],
    CONFIG["image_size"],
    CONFIG["encoder"],
    CONFIG["encoder_weights"],
    CONFIG["frequency_aug_p"],
    CONFIG["frequency_beta_range"],
    CONFIG["frequency_blend_range"]
)

model = RoadModel(
    encoder=CONFIG["encoder"],
    encoder_weights=CONFIG["encoder_weights"],
    in_channels=3,
    out_classes=CONFIG["classes"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    epochs=CONFIG["epochs"]
)

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    monitor="val_miou",
    dirpath="./checkpoints",
    filename="segformer-b2-E3-cityscapes-{epoch:02d}-{val_miou:.2f}",
    save_top_k=1,
    mode="max",
)

trainer = pl.Trainer(
    max_epochs=CONFIG["epochs"],
    accelerator="gpu",
    devices=1,
    logger=True,
    callbacks=[
        checkpoint_callback,
        TQDMProgressBar(refresh_rate=20),
        ClearMLMetricsCallback(task)
    ],
    precision="16-mixed",
    accumulate_grad_batches=4
)

trainer.fit(model, datamodule=dm)

In [ ]:
best_model_path = trainer.checkpoint_callback.best_model_path

best_model = RoadModel.load_from_checkpoint(best_model_path)
best_model.to(CONFIG["device"])
best_model.eval()


val_results = trainer.validate(best_model, datamodule=dm)[0]

metrics_data = {
    "Metric": ["Loss", "mIoU", "Percent"],
    "Value": [
        f"{val_results['val_loss']:.4f}",
        f"{val_results['val_miou']:.4f}",
        f"{val_results['val_miou']*100:.2f}%"
    ]
}
df = pd.DataFrame(metrics_data)

print("\n" + "="*30)
print("Results:" + "\n")
print("="*30)
display(df)

def visualize_predictions(model, datamodule, num_samples=3):
    model.to(CONFIG["device"])
    model.eval()

    val_loader = datamodule.val_dataloader()
    images, masks = next(iter(val_loader))

    images = images.to(CONFIG["device"])
    with torch.no_grad():
        logits = model(images)
        preds = torch.argmax(logits, dim=1)

    images = images.cpu()
    masks = masks.cpu()
    preds = preds.cpu()

    plt.figure(figsize=(18, num_samples * 4))

    for i in range(num_samples):
        img = images[i].permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())

        plt.subplot(num_samples, 3, i*3 + 1)
        plt.imshow(img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(num_samples, 3, i*3 + 2)
        plt.imshow(masks[i])
        plt.title("Ground truth")
        plt.axis("off")

        plt.subplot(num_samples, 3, i*3 + 3)
        plt.imshow(preds[i])
        plt.title(f"Prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

visualize_predictions(best_model, dm)

In [ ]:
best_model_path = trainer.checkpoint_callback.best_model_path

model_name = os.path.basename(best_model_path)

current_task = Task.get_task(task_id=task.id)

current_task.update_output_model(
    model_path=best_model_path,
    model_name=model_name,
    iteration=trainer.current_epoch
)

In [ ]:
task.close()

In [ ]:
from google.colab import runtime
runtime.unassign()